# PySpark DataFrame Joins

## What This Notebook Demonstrates
This notebook teaches you how to combine data from multiple DataFrames using joins. You'll learn join syntax, different join types, and how to handle edge cases like NULL keys and column name collisions.

---

## Key Concepts Explained

### **1. Creating Sample Data with Edge Cases**

```python
rows_customers = [
    (1, "Asha", "IN", True),
    (None, "Ghost", "UK", False),  # NULL key
]
```

**Concepts:**
* **List of tuples** - Each tuple represents one row of data
  * Order of values must match schema order
  * Use `None` for NULL values in Python
* **Intentional edge cases in test data:**
  * Customer with `NULL` customer_id ("Ghost") - tests NULL join behavior
  * Order with customer_id=5 - no matching customer (tests unmatched records)
  * Order with `NULL` customer_id - tests NULL handling
  * Both tables have "country" column - tests column name collision

**Why include edge cases?**
* ✅ Understand how joins handle NULLs (they DON'T match)
* ✅ See what happens with unmatched records
* ✅ Learn column disambiguation techniques
* ✅ Write more robust production code

---

### **2. Defining Schema with StructType**

```python
schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("vip", T.BooleanType(), True),
])
```

**Concepts:**
* **`T.StructType([ ... ])`** - Container for the entire DataFrame schema
  * Must import as `from pyspark.sql import types as T`
  * Takes a list of StructField objects
* **`T.StructField(name, dataType, nullable)`** - Defines one column
  * **name** (string) - Column name
  * **dataType** - Data type (see table below)
  * **nullable** (boolean) - Can this column contain NULL values?
    * `True` = allows NULLs (default, most common)
    * `False` = NULLs not allowed (rare, for required fields)

**Data Types Used:**

| PySpark Type | Python Type | Example | Use Case |
|--------------|-------------|---------|----------|
| `IntegerType()` | int | 42 | IDs, counts, whole numbers |
| `StringType()` | str | "Alice" | Text, names, codes |
| `DoubleType()` | float | 19.99 | Prices, measurements, decimals |
| `BooleanType()` | bool | True/False | Flags, yes/no values |

---

### **3. Creating DataFrames from Data**

```python
df_customers = spark.createDataFrame(rows_customers, schema_customers)
```

**Concepts:**
* **`spark.createDataFrame(data, schema)`** - Creates DataFrame from Python data
  * **data** - List of tuples/rows
  * **schema** - StructType defining structure
* **When to use:**
  * Testing with sample data (like this notebook)
  * Converting small Python datasets to Spark
  * Unit tests
* **Production alternative:**
  * Usually read from files: `spark.read.csv()`, `spark.read.parquet()`

---

### **4. The Join Operation - Combining DataFrames**

```python
df_inner = df_orders.join(df_customers, on="customer_id", how="inner")
```

**Concepts:**
* **`df1.join(df2, on, how)`** - Combines two DataFrames
  * **df1** (left side) - The DataFrame you call `.join()` on
  * **df2** (right side) - The DataFrame passed as parameter
  * **on** - Join key(s) - column(s) to match on
  * **how** - Join type (see below)

#### **Join Parameters Explained:**

**`on` parameter - Join Keys:**
```python
# Single column (both DataFrames have same column name)
on="customer_id"

# Multiple columns (natural join)
on=["customer_id", "country"]

# Different column names (explicit condition)
on=df_orders.customer_id == df_customers.cust_id
```

**`how` parameter - Join Types:**

| Join Type | Keyword | Returns | Use Case |
|-----------|---------|---------|----------|
| **Inner** | `"inner"` | Only matching rows from both sides | Most common; find related records |
| **Left (Outer)** | `"left"` or `"left_outer"` | All left + matching right (NULLs for non-matches) | Keep all primary records, add optional details |
| **Right (Outer)** | `"right"` or `"right_outer"` | All right + matching left (NULLs for non-matches) | Less common (just swap tables and use left) |
| **Full (Outer)** | `"full"` or `"full_outer"` | All rows from both (NULLs for non-matches) | Find all records, matched or not |
| **Left Semi** | `"left_semi"` | Left rows that HAVE matches in right | Filtering: "customers who placed orders" |
| **Left Anti** | `"left_anti"` | Left rows that DON'T have matches in right | Filtering: "customers who never ordered" |
| **Cross** | `"cross"` | Cartesian product (all combinations) | Rare; combinatorics problems |

---

### **5. Inner Join - Only Matching Records**

```python
df_inner = df_orders.join(df_customers, on="customer_id", how="inner")
```

**How it works:**
1. **Looks at join key** (customer_id) in both DataFrames
2. **Finds matches** - Rows where customer_id values are equal
3. **Combines columns** - Creates new row with columns from both DataFrames
4. **Drops non-matches** - Excludes rows without matching key

**What gets EXCLUDED in inner join:**
* ❌ Customer with `NULL` customer_id ("Ghost") - NULLs never match
* ❌ Order with customer_id=5 - No matching customer
* ❌ Order with `NULL` customer_id - NULLs never match
* ❌ Customer Diana (id=4) - No orders placed

**What gets INCLUDED:**
* ✅ Asha (customer_id=1) - Has 2 orders → 2 result rows
* ✅ Bob (customer_id=2) - Has 2 orders → 2 result rows
* ✅ Chen (customer_id=3) - Has 2 orders → 2 result rows

**Result:** 6 rows total (only customers who placed orders)

---

### **6. NULL Behavior in Joins** ⚠️ **Critical Concept**

**Rule: NULL values NEVER match in joins (even NULL = NULL is false)**

```
Customer: customer_id = NULL ("Ghost")
Order:    customer_id = NULL
Result:   These DO NOT join together!
```

**Why?** 
* In SQL logic, `NULL` means "unknown value"
* You can't determine if two unknown values are equal
* Therefore: `NULL = NULL` evaluates to `NULL` (not true)

**Practical implications:**
* Always handle NULL keys before joining (filter out or replace with default)
* NULL customer_id in orders = orphaned records
* Use `LEFT JOIN` to find NULLs: non-matched rows get NULL in joined columns

---

### **7. Column Name Collisions**

Both DataFrames have a "country" column. After joining:
* Spark keeps BOTH columns
* Result has **two** "country" columns (one from orders, one from customers)

**How to handle collisions:**

```python
# Method 1: Select specific columns after join
df_inner.select(
    "order_id", 
    "customer_id", 
    df_orders.country.alias("order_country"),
    df_customers.country.alias("customer_country")
)

# Method 2: Rename before joining
df_orders_renamed = df_orders.withColumnRenamed("country", "order_country")
df_customers_renamed = df_customers.withColumnRenamed("country", "customer_country")

# Method 3: Use table aliases in join condition
df_orders.alias("o").join(
    df_customers.alias("c"),
    F.col("o.customer_id") == F.col("c.customer_id")
)
```

---

### **8. Important Functions Summary**

| Function | Purpose | Example |
|----------|---------|----------|
| `spark.createDataFrame(data, schema)` | Create DataFrame from Python data | Testing, small datasets |
| `T.StructType([...])` | Define DataFrame schema | Structure definition |
| `T.StructField(name, type, nullable)` | Define single column | Column specification |
| `df.join(other, on, how)` | Combine two DataFrames | Merge related data |
| `display(df)` | Show DataFrame in table format | Data preview |

---

## Join Selection Guide

**Choose your join type based on what you want to keep:**

```
┌─────────────┐         ┌─────────────┐
│  Customers  │         │   Orders    │
│  (4 rows)   │         │  (8 rows)   │
└─────────────┘         └─────────────┘
       ↓                       ↓
       └───────── JOIN ─────────┘
                  ↓
         Which rows to keep?

• Inner  → Only matched (6 rows)
• Left   → All customers + matched orders (8 rows)
• Right  → All orders + matched customers (8 rows)
• Full   → Everything (10 rows with NULLs)
• Semi   → Customers who ordered (3 rows, customer cols only)
• Anti   → Customers who didn't order (1 row, customer cols only)
```

---

## Best Practices Demonstrated

1. **Use explicit schemas** with StructType for clarity and type safety
2. **Include edge cases in test data** (NULLs, non-matches) to verify join logic
3. **Specify join type explicitly** (`how="inner"`) - don't rely on defaults
4. **Handle NULL keys** before joining (filter or coalesce to default value)
5. **Watch for column name collisions** - rename columns proactively
6. **Choose the right join type** - inner for matching only, left for keeping primary table
7. **Test with small sample data** before running on production datasets

---

## Common Join Mistakes to Avoid

❌ **Assuming NULL = NULL** (they don't match!)  
❌ **Forgetting to specify join type** (default varies by context)  
❌ **Not handling duplicate column names** (causes confusion later)  
❌ **Using CROSS join accidentally** (explodes row count)  
❌ **Not validating unmatched records** (data quality issues)  

---

## Next Steps

This notebook shows **inner join**. Try experimenting with:
* `how="left"` - See Diana (no orders) included with NULL order columns
* `how="full"` - See ALL records with NULLs for non-matches
* `how="left_anti"` - Find customers who never ordered (Diana)
* Adding more join examples for other join types

# Joins

In [0]:
from pyspark.sql import functions as F, types as T

rows_customers = [
    (1,  "Asha",  "IN", True),
    (2,  "Bob",   "US", False),
    (3,  "Chen",  "CN", True),
    (4,  "Diana", "US", None),
    (None, "Ghost","UK", False),     # NULL key to demo null join behavior
]

rows_orders = [
    (101, 1,   120.0, "IN"),
    (102, 1,    80.0, "IN"),
    (103, 2,    50.0, "US"),
    (104, 5,    30.0, "DE"),         # no matching customer_id
    (105, 3,   200.0, "CN"),
    (106, None, 15.0, "UK"),         # NULL key won’t match
    (107, 3,    40.0, "CN"),
    (108, 2,    75.0, "US"),
]

schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("vip",         T.BooleanType(), True),
])

schema_orders = T.StructType([
    T.StructField("order_id",    T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount",      T.DoubleType(),  True),
    T.StructField("country",     T.StringType(),  True),  # same column name to show collisions
])

df_customers = spark.createDataFrame(rows_customers, schema_customers)
df_orders    = spark.createDataFrame(rows_orders,    schema_orders)

display(df_customers)
display(df_orders)

customer_id,name,country,vip
1,Asha,IN,true
2,Bob,US,false
3,Chen,CN,true
4,Diana,US,null
null,Ghost,UK,false


order_id,customer_id,amount,country
101,1,120.0,IN
102,1,80.0,IN
103,2,50.0,US
104,5,30.0,DE
105,3,200.0,CN
106,null,15.0,UK
107,3,40.0,CN
108,2,75.0,US


In [0]:
df_inner = df_orders.join(df_customers, on="customer_id", how="inner")
display(df_inner)

customer_id,order_id,amount,country,name,country,vip
1,101,120.0,IN,Asha,IN,true
1,102,80.0,IN,Asha,IN,true
2,103,50.0,US,Bob,US,false
3,105,200.0,CN,Chen,CN,true
3,107,40.0,CN,Chen,CN,true
2,108,75.0,US,Bob,US,false
